In [1]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.1 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "train.jsonl",
        "validation": "validation.jsonl",
        "test": "test.jsonl"
    }
)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 326
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 41
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 41
    })
})

In [4]:
model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

In [6]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [8]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [10]:
dataset["train"][0]

{'messages': [{'role': 'system',
   'content': 'You are a legal contract analysis assistant. Extract important clauses and return JSON.'},
  {'role': 'user',
   'content': 'Exhibit 10.28 PRODUCT SALE AND MARKETING AGREEMENT THIS PRODUCT SALE AND MARKETING AGREEMENT (this "Agreement") is made this 12th day of November, 2018 (the "Effective Date"), by and between Calm.com, Inc., a Delaware corporation, having offices at 140 2nd Street, 3rd Floor, San Francisco, California 94105 ("Calm") and XpresSpa Group, Inc., a Delaware corporation, having offices at 780 Third Avenue, 12th Floor, New York, New York 10017 ("XSPA"). Each of Calm and XSPA may be referred to herein individually as a "Party" and collectively as the "Parties". RECITALS WHEREAS, Calm is the manufacturer and distributor of Calm branded products and services, including those set forth on Exhibit A (the "Products"); WHEREAS, XSPA is the owner, operator and/or franchisor of XpresSpa branded stores (each a "Store") throughout the

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/capstone/data"

TRAIN_FILE = f"{DATA_DIR}/train.jsonl"
VAL_FILE = f"{DATA_DIR}/validation.jsonl"
TEST_FILE = f"{DATA_DIR}/test.jsonl"

MODEL_OUTPUT = "/content/drive/MyDrive/Colab Notebooks/capstone/models/cuad-clause-extractor-v1"

In [13]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": TRAIN_FILE,
        "validation": VAL_FILE,
        "test": TEST_FILE
    }
)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 326
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 41
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 41
    })
})

In [14]:
model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"

In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

In [16]:
!pip install -U bitsandbytes

In [17]:
model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"

In [18]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [19]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [20]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully


In [21]:
dataset["train"].column_names

['messages']

In [22]:
dataset["train"][0]

{'messages': [{'role': 'system',
   'content': 'You are a legal contract analysis assistant. Extract important clauses and return JSON.'},
  {'role': 'user',
   'content': 'Exhibit 10.28 PRODUCT SALE AND MARKETING AGREEMENT THIS PRODUCT SALE AND MARKETING AGREEMENT (this "Agreement") is made this 12th day of November, 2018 (the "Effective Date"), by and between Calm.com, Inc., a Delaware corporation, having offices at 140 2nd Street, 3rd Floor, San Francisco, California 94105 ("Calm") and XpresSpa Group, Inc., a Delaware corporation, having offices at 780 Third Avenue, 12th Floor, New York, New York 10017 ("XSPA"). Each of Calm and XSPA may be referred to herein individually as a "Party" and collectively as the "Parties". RECITALS WHEREAS, Calm is the manufacturer and distributor of Calm branded products and services, including those set forth on Exhibit A (the "Products"); WHEREAS, XSPA is the owner, operator and/or franchisor of XpresSpa branded stores (each a "Store") throughout the

In [23]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully


In [24]:
def format_example(example):
    messages = example["messages"]

    prompt = ""

    for message in messages:
        if message["role"] == "user":
            prompt += "### Instruction:\n"
            prompt += message["content"]
            prompt += "\n\n"

        elif message["role"] == "assistant":
            prompt += "### Response:\n"
            prompt += message["content"]

    return {
        "text": prompt
    }

In [25]:
formatted_dataset = dataset.map(
    format_example
)

formatted_dataset["train"][0]

Map:   0%|          | 0/326 [00:00<?, ? examples/s]

Map:   0%|          | 0/41 [00:00<?, ? examples/s]

Map:   0%|          | 0/41 [00:00<?, ? examples/s]

{'messages': [{'role': 'system',
   'content': 'You are a legal contract analysis assistant. Extract important clauses and return JSON.'},
  {'role': 'user',
   'content': 'Exhibit 10.28 PRODUCT SALE AND MARKETING AGREEMENT THIS PRODUCT SALE AND MARKETING AGREEMENT (this "Agreement") is made this 12th day of November, 2018 (the "Effective Date"), by and between Calm.com, Inc., a Delaware corporation, having offices at 140 2nd Street, 3rd Floor, San Francisco, California 94105 ("Calm") and XpresSpa Group, Inc., a Delaware corporation, having offices at 780 Third Avenue, 12th Floor, New York, New York 10017 ("XSPA"). Each of Calm and XSPA may be referred to herein individually as a "Party" and collectively as the "Parties". RECITALS WHEREAS, Calm is the manufacturer and distributor of Calm branded products and services, including those set forth on Exhibit A (the "Products"); WHEREAS, XSPA is the owner, operator and/or franchisor of XpresSpa branded stores (each a "Store") throughout the

In [26]:
def tokenize(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        max_length=2048,
        padding="max_length"
    )

    result["labels"] = result["input_ids"].copy()

    return result


tokenized_dataset = formatted_dataset.map(
    tokenize,
    batched=True,
    remove_columns=formatted_dataset["train"].column_names
)

tokenized_dataset

Map:   0%|          | 0/326 [00:00<?, ? examples/s]

Map:   0%|          | 0/41 [00:00<?, ? examples/s]

Map:   0%|          | 0/41 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 326
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 41
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 41
    })
})

In [27]:
tokenized_dataset["train"][0].keys()

dict_keys(['input_ids', 'attention_mask', 'labels'])

In [28]:
!pip install -U peft

In [29]:
from peft import LoraConfig, get_peft_model

In [30]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "v_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 819,200 || all params: 362,640,320 || trainable%: 0.2259


In [31]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    report_to="none"
)

In [32]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"]
)

In [33]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.446655,2.191729
2,2.068762,2.016767
3,2.191165,2.005667


TrainOutput(global_step=246, training_loss=2.2850169980429054, metrics={'train_runtime': 1441.1264, 'train_samples_per_second': 0.679, 'train_steps_per_second': 0.171, 'total_flos': 3791024986521600.0, 'train_loss': 2.2850169980429054, 'epoch': 3.0})

In [34]:
model.save_pretrained(MODEL_OUTPUT)
tokenizer.save_pretrained(MODEL_OUTPUT)

('/content/drive/MyDrive/Colab Notebooks/capstone/models/cuad-clause-extractor-v1/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/capstone/models/cuad-clause-extractor-v1/chat_template.jinja',
 '/content/drive/MyDrive/Colab Notebooks/capstone/models/cuad-clause-extractor-v1/tokenizer.json')

In [35]:
import os

print(os.listdir(MODEL_OUTPUT))

['checkpoint-82', 'checkpoint-164', 'checkpoint-246', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']
